In [ ]:
import pandas as pd
import scanpy as sc
import rapids_singlecell as rsc


In [ ]:
adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')
adata_qc

In [ ]:
adata_epi = sc.read_h5ad('./adata_epi.h5ad')
adata_epi

In [ ]:
celltype_dict = adata_epi.obs['cell_subtype'].to_dict()
celltype_dict

In [ ]:
adata_qc.obs['cell_subtype'] = adata_qc.obs.index.map(celltype_dict)

In [ ]:
adata = adata_qc[adata_qc.obs['cell_subtype']=='Epi_CA9',:].copy()
adata

In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=50)


In [ ]:
rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=40)

In [ ]:
rsc.pp.neighbors(adata, n_neighbors=30, n_pcs=10, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)

In [ ]:
rsc.tl.leiden(adata, resolution=0.2, key_added='leiden_0.2_detailed')

In [ ]:
adata.obs['leiden_0.2_detailed'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_0.2_detailed', method='t-test')

In [ ]:
result = adata.uns['rank_genes_groups']
all_top_genes = []
for group in result['names'].dtype.names:
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    top_50_genes = group_data.head(100)['gene'].tolist()
    all_top_genes.append({group: top_50_genes})
    group_data.to_csv(f'./subtype_degs/epi_ca9_subcluters_deg_{group}_filtered_ttest.csv', index=False)
    print(f"Top 100 DEGs in cluster {group}:")
    gene_list = result['names'][group][:100]
    print(", ".join(gene_list))
    
# Print the top 100 DEGs for all groups if needed.
#print(all_top_genes)



In [ ]:
anno_dict = {
    '0': 'Epi_CD74',     
    '1': 'Epi_FTL',       
    '2': 'Epi_AHNAK',  
    '3': 'Epi_NAT8', }      
adata.obs['cell_subtype'] = adata.obs['leiden_0.2_detailed'].map(anno_dict)

In [ ]:
adata.write_h5ad('adata_epi_ca9.h5ad')

In [ ]:
from pathlib import Path
import matplotlib as mpl

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
sc.set_figure_params(figsize=(3, 3), dpi=150)

base_dir = Path(".")
fig_dir = base_dir / "figures"
fig_dir.mkdir(exist_ok=True)
sc.settings.figdir = str(fig_dir)

adata = sc.read_h5ad(base_dir / "adata_epi_ca9.h5ad")
adata.obs["leiden_0.2_detailed"] = adata.obs["leiden_0.2_detailed"].astype(str)
leiden_0p2_order = sorted(
    adata.obs["leiden_0.2_detailed"].unique(),
    key=lambda x: int(x) if x.isdigit() else x,
)

representative_top3_markers_no_cd44 = {
    "Epi_MIOX": ["MIOX", "PCK1", "NAT8"],
    "Epi_GPX3": ["GPX3", "GATM", "ASS1"],
    "Epi_ALDOB": ["ALDOB", "FABP1", "APOE"],
    "Epi_CA9": ["CA9", "NDUFA4L2", "EGLN3"],
    "Epi_JUN": ["JUN", "FOS", "ATF3"],
    "Epi_VIM": ["VIM", "S100A10", "LGALS1"],
}


In [ ]:
sc.pl.dotplot(
    adata,
    var_names=representative_top3_markers_no_cd44,
    groupby="leiden_0.2_detailed",
    standard_scale="var",
    use_raw=True,
    categories_order=leiden_0p2_order,
    save="_ca9_recluster_representative_top3_no_cd44_no_aqp2_ca12_by_leiden0p2_text.pdf",
)


In [ ]:
sc.pl.umap(
    adata,
    color="leiden_0.2_detailed",
    save="_ca9_recluster_leiden0p2_text.pdf",
)
